# Agentic RAG — Taller de Tecnologías 2

Chatbot en lenguaje natural que responde sobre un dataset de correos de phishing y sobre documentos
largos, con un agente de LangGraph, tool calling y memoria a corto y largo plazo.

Este notebook importa los módulos de `src/` y muestra el sistema funcionando de punta a punta. Integra:

| Componente | Elección |
|---|---|
| LLM | `qwen/qwen3.8-27b` vía Groq |
| Embeddings | `intfloat/multilingual-e5-small`, locales |
| Base vectorial | ChromaDB persistente, colecciones `dataset` y `papers` |
| Agente | Grafo de LangGraph escrito a mano, sin `create_react_agent` ni `ToolNode` |
| Memoria | `SqliteSaver` (corto plazo) y `SqliteStore` (largo plazo) |

**Requisitos previos:** seguir los pasos de `README.md` para tener `.env`, `data/` y `chroma_db/`.

## 1. Preparación

In [1]:
import builtins
import sys
import tempfile
import time
import uuid
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_text_splitters import RecursiveCharacterTextSplitter

sys.path.insert(0, str(Path.cwd()))

from src import agente as modulo_agente
from src import chatbot, config, datos, ingesta, memoria, recuperacion

# En False se omiten las celdas que consumen tokens de Groq.
EJECUTAR_LLM = True

# Memoria temporal: cada ejecución del notebook arranca limpia y no toca memoria.sqlite3.
config.ARCHIVO_MEMORIA = Path(tempfile.mkdtemp()) / "memoria_notebook.sqlite3"
pd.set_option("display.max_colwidth", 120)

print("Clave de Groq configurada:", bool(config.GROQ_API_KEY))
print("Papers disponibles:", len(list(config.DIR_PAPERS.glob("*.pdf"))))
print("Dataset consolidado:", config.CSV_CONSOLIDADO.exists())
print("Base vectorial:", config.DIR_CHROMA.exists())

Clave de Groq configurada: True
Papers disponibles: 6
Dataset consolidado: True
Base vectorial: True


## 2. Dataset

Se eligió el [Phishing Email Dataset](https://www.kaggle.com/datasets/naserabdullahalam/phishing-email-dataset).
Los datos están en inglés y el chatbot funciona en español e inglés. `src/datos.py` une seis corpus en un
único CSV y conserva el corpus de origen, el remitente, el asunto, el cuerpo y la etiqueta.

In [2]:
correos = datos.cargar_consolidado()
print(f"Correos: {len(correos):,}")
correos[["fuente", "sender", "subject", "urls", "label"]].head()

Correos: 82,486


,fuente,sender,subject,urls,label
0,CEAS_08,Young Esposito <Young@iworld.de>,Never agree to be a loser,1.0,1
1,CEAS_08,Mok <ipline's1983@icable.ph>,Befriend Jenna Jameson,1.0,1
2,CEAS_08,Daily Top 10 <Karmandeep-opengevl@universalnet.psi.br>,CNN.com Daily Top 10,1.0,1
3,CEAS_08,Michael Parker <ivqrnai@pobox.com>,Re: svn commit: r619753 - in /spamassassin/trunk: lib/Mail/SpamAssassin/PerMsgStatus.pm lib/Mail/SpamAssassin/Util/R...,1.0,0
4,CEAS_08,Gretchen Suggs <externalsep1@loanofficertool.com>,SpecialPricesPharmMoreinfo,1.0,1


In [3]:
composicion = correos.groupby("fuente")["label"].agg(total="count", phishing="sum")
composicion["legitimos"] = composicion["total"] - composicion["phishing"]
composicion["remitente_disponible"] = correos.groupby("fuente")["sender"].apply(lambda serie: serie.notna().mean().round(2))
composicion.sort_values("total", ascending=False)

,total,phishing,legitimos,remitente_disponible
fuente,,,,
CEAS_08,39154,21842,17312,1.0
Enron,29767,13976,15791,0.0
SpamAssasin,5809,1718,4091,1.0
Nigerian_Fraud,3332,3332,0,0.9
Ling,2859,458,2401,0.0
Nazario,1565,1565,0,1.0


Enron y Ling no incluyen remitente ni URLs, por eso las estadísticas informan sobre cuántos correos se
calcularon. Además, la etiqueta `phishing` del dataset también incluye spam común.

Para la búsqueda semántica se indexa una muestra balanceada de 10.000 correos, por indicación docente.
Las estadísticas se calculan sobre el dataset completo.

## 3. Chunking y embeddings

Los textos se cortan con `RecursiveCharacterTextSplitter` (800 caracteres, 100 de solapamiento), que
prioriza respetar párrafos y oraciones. Cada fragmento conserva sus metadatos para citar y filtrar.

In [4]:
separador = RecursiveCharacterTextSplitter(chunk_size=config.CHUNK_SIZE, chunk_overlap=config.CHUNK_OVERLAP)
correo_largo = correos.loc[correos["body"].astype(str).str.len().between(2000, 3000)].iloc[0]
fragmentos = separador.split_text(str(correo_largo["body"]))

print(f"Correo de {len(str(correo_largo['body']))} caracteres -> {len(fragmentos)} fragmentos")
print("Largo de cada fragmento:", [len(fragmento) for fragmento in fragmentos])

Correo de 2040 caracteres -> 4 fragmentos
Largo de cada fragmento: [740, 753, 212, 415]


Los PDF se normalizan y se descartan los fragmentos con menos de 50% de letras. Esos fragmentos suelen
ser tablas numéricas, y en las pruebas aparecían como resultado de consultas sin relación.

In [5]:
print(repr(ingesta.normalizar_texto("El Trans-\nformer usa   atención\nmulti-cabeza.")))
print("Texto común:", ingesta.es_fragmento_util("The Transformer relies entirely on attention mechanisms."))
print("Tabla numérica:", ingesta.es_fragmento_util("28.4 41.8 2.3 1.0 3.3 10 18 5.0 7.7 9.2 26.3"))

'El Transformer usa atención multi-cabeza.'
Texto común: True
Tabla numérica: False


El modelo e5 es multilingüe: textos con el mismo significado quedan cerca aunque estén en otro idioma.
Esto permite preguntar en español sobre documentos en inglés. El modelo exige los prefijos `query:` y
`passage:`, que agrega la clase `EmbeddingsE5`.

In [6]:
embeddings = ingesta.crear_embeddings()
consulta = np.array(embeddings.embed_query("correo que pide verificar la contraseña de la cuenta"))
textos = [
    "Your password has expired, click here to verify your account",
    "The quarterly meeting has been moved to Tuesday afternoon",
    "Add the tomatoes and let the sauce simmer for ten minutes",
]
vectores = np.array(embeddings.embed_documents(textos))
similitud = vectores @ consulta / (np.linalg.norm(vectores, axis=1) * np.linalg.norm(consulta))

print("Dimensión del vector:", consulta.shape[0])
pd.DataFrame({"texto en inglés": textos, "similitud coseno": similitud.round(3)})

Dimensión del vector: 384


,texto en inglés,similitud coseno
0,"Your password has expired, click here to verify your account",0.815
1,The quarterly meeting has been moved to Tuesday afternoon,0.716
2,Add the tomatoes and let the sauce simmer for ten minutes,0.721


## 4. Base vectorial y búsqueda semántica

Los índices se construyen una vez con `scripts/construir_indice.py` y `scripts/construir_indice_papers.py`,
y quedan persistidos en `chroma_db/`.

In [7]:
for coleccion in (config.COLECCION_DATASET, config.COLECCION_PAPERS):
    print(f"{coleccion}: {ingesta.cargar_indice(coleccion)._collection.count():,} fragmentos")

dataset: 33,792 fragmentos
papers: 2,936 fragmentos


In [8]:
consultas = {
    "¿qué es el mecanismo de atención multi-cabeza?": "pertinente",
    "¿cómo escala la pérdida con el tamaño del modelo?": "pertinente",
    "replicación líder-seguidor en bases de datos": "pertinente",
    "mejores playas de Uruguay para el verano": "ajena",
    "¿cuál es la capital de Australia?": "ajena",
}
indice_papers = ingesta.cargar_indice(config.COLECCION_PAPERS)
filas = []
for consulta_texto, tipo in consultas.items():
    documento, puntaje = indice_papers.similarity_search_with_relevance_scores(consulta_texto, k=1)[0]
    filas.append({
        "consulta": consulta_texto,
        "tipo": tipo,
        "relevancia": round(puntaje, 3),
        "supera el umbral": puntaje >= config.UMBRAL_RELEVANCIA,
        "mejor resultado": f"{documento.metadata['titulo'][:40]}, pág. {documento.metadata['pagina']}",
    })
pd.DataFrame(filas)

,consulta,tipo,relevancia,supera el umbral,mejor resultado
0,¿qué es el mecanismo de atención multi-cabeza?,pertinente,0.715,True,"Attention Is All You Need, pág. 4"
1,¿cómo escala la pérdida con el tamaño del modelo?,pertinente,0.767,True,"Scaling Laws for Neural Language Models, pág. 22"
2,replicación líder-seguidor en bases de datos,pertinente,0.806,True,"Designing Data-Intensive Applications (M, pág. 174"
3,mejores playas de Uruguay para el verano,ajena,0.637,False,"Designing Data-Intensive Applications (M, pág. 76"
4,¿cuál es la capital de Australia?,ajena,0.635,False,"Designing Data-Intensive Applications (M, pág. 572"


Los puntajes de e5 están comprimidos y se solapan, por lo que un umbral no separa por sí solo lo pertinente
de lo ajeno. El umbral es un piso: si nada lo supera, la tool devuelve `SIN EVIDENCIA`. El resto del control
lo hace el prompt, que exige verificar que los fragmentos respondan la pregunta. Por ejemplo, "receta de lasaña
vegetariana" supera el umbral con 0,72 y recupera un fragmento sin relación, por lo que el umbral solo no alcanza.

## 5. Herramientas del agente

El agente recibe estas tools. Los ejemplos siguientes muestran lo que devuelven, sin usar el LLM.

In [9]:
herramientas = recuperacion.HERRAMIENTAS + [memoria.guardar_recuerdo]
pd.DataFrame([
    {"tool": herramienta.name, "descripción": herramienta.description.split("\n")[0]}
    for herramienta in herramientas
])

,tool,descripción
0,buscar_en_papers,"Busca fragmentos en los papers/documentos cargados (Transformers/Attention,"
1,buscar_en_dataset,Busca correos parecidos en el dataset de correos de phishing y legítimos.
2,listar_papers,Devuelve la lista de documentos/papers disponibles para consultar.
3,estadisticas_dataset,"Devuelve estadísticas del dataset de correos: total, cantidad de phishing y"
4,ranking_dataset,Devuelve los valores más frecuentes del dataset completo. Usar para preguntas
5,guardar_recuerdo,Guarda en la memoria a largo plazo un dato duradero sobre el usuario


In [10]:
print(recuperacion.buscar_en_papers.invoke({"consulta": "multi-head attention", "titulo_paper": "attention", "cantidad": 1})[:500])
print("\n--- Consulta ajena ---")
print(recuperacion.buscar_en_papers.invoke({"consulta": "mejores playas de Uruguay para el verano"})[:300])
print("\n--- Correos ---")
print(recuperacion.buscar_en_dataset.invoke({"consulta": "account suspended verify password", "tipo_correo": "phishing"})[:500])

[1] Fuente: Attention Is All You Need, pág. 5 (relevancia 0.83)
output values. These are concatenated and once again projected, resulting in the final values, as depicted in Figure 2. Multi-head attention allows the model to jointly attend to information from different representation subspaces at different positions. With a single attention head, averaging inhibits this. MultiHead(Q, K, V) = Concat(head1, ...,headh)W O where headi = Attention(QW Qi , KWKi , V WVi ) Where the projections are para

--- Consulta ajena ---
SIN EVIDENCIA: no se encontraron fragmentos relevantes para esta consulta.

--- Correos ---


[1] Fuente: correo PHISHING | corpus Enron (relevancia 0.83)
security
assistance
it has come to our attention that your account information needs to be
updated as part of our continuing commitment to protect your account and to
reduce the instance of fraud on our website . just update your personal records and you will not run into
any future problems with the online service .
however , failure to update your records until july . 21 , 2005 your account considered as suspension .
please up


In [11]:
print(recuperacion.estadisticas_dataset.invoke({}))
print()
print(recuperacion.ranking_dataset.invoke({"campo": "dominio", "tipo_correo": "phishing", "cantidad": 5}))

Total de correos: 82486
Phishing: 42891 | Legítimos: 39595
Correos indexados para búsqueda semántica: muestra balanceada de 10000
Correos con el dato de URLs: 49860 (Enron y Ling no lo tienen)
Con URLs entre esos correos: 33806 (phishing: 18857, legítimos: 14949)
Por corpus de origen (total / phishing):
  - CEAS_08: 39154 / 21842
  - Enron: 29767 / 13976
  - Ling: 2859 / 458
  - Nazario: 1565 / 1565
  - Nigerian_Fraud: 3332 / 3332
  - SpamAssasin: 5809 / 1718

Calculado sobre 27139 de 42891 correos (el resto no tiene este dato):
- hotmail.com: 519
- yahoo.com: 492
- google.com: 209
- virgilio.it: 159
- netscape.net: 130


## 6. Agente con LangGraph

El grafo tiene dos nodos escritos a mano. `agente` llama al LLM con las tools disponibles y decide si
responde o si pide herramientas. `herramientas` las ejecuta y devuelve el control. No hay un router aparte:
la clasificación entre documentos, consulta general y small-talk la decide el LLM guiado por el prompt.

In [12]:
agente = modulo_agente.construir_agente(memoria.crear_memoria_corto_plazo(), memoria.crear_memoria_largo_plazo())
print(agente.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	agente(agente)
	herramientas(herramientas)
	__end__([<p>__end__</p>]):::last
	__start__ --> agente;
	agente -.-> __end__;
	agente -.-> herramientas;
	herramientas --> agente;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



### Prompt de sistema

Técnicas aplicadas: rol, instrucciones por tipo de consulta, reglas contra alucinaciones, citas obligatorias,
ejemplos few-shot, idioma espejo y límite de longitud.

In [13]:
print(modulo_agente.PROMPT_SISTEMA)

Sos un asistente experto en ciberseguridad (phishing) y en los papers de la materia.
Tenés acceso a herramientas:
- buscar_en_papers / listar_papers: para preguntas sobre los documentos cargados (Attention, GPT, GPT-3, Scaling Laws, Subliminal Learning, Designing Data-Intensive Applications).
- buscar_en_dataset: para mostrar ejemplos reales de correos de phishing o legítimos, o comparar un correo sospechoso con casos conocidos.
- estadisticas_dataset: para preguntas de cantidades o composición del dataset.
- ranking_dataset: para saber qué remitentes, dominios o asuntos aparecen más.
- guardar_recuerdo: para recordar datos del usuario en futuras conversaciones.

Cómo decidir según el tipo de consulta:
1. Saludos, agradecimientos o charla (small-talk): respondé breve y amable, SIN usar herramientas.
2. Preguntas sobre los papers o el dataset: usá SIEMPRE la herramienta correspondiente antes de responder, también en las preguntas de seguimiento. Los fragmentos de turnos anteriores ya no

## 7. Con contexto y sin contexto

Esta prueba muestra el aporte del RAG con la misma pregunta. Primero el LLM responde sin documentos y
después con los fragmentos recuperados.

In [14]:
pregunta = "¿Qué condición deben cumplir el modelo maestro y el modelo estudiante para que ocurra el subliminal learning?"

if EJECUTAR_LLM:
    llm = modulo_agente.crear_llm()
    sin_contexto = modulo_agente.invocar_llm(llm, [HumanMessage(pregunta + " Respondé en dos oraciones.")])
    contexto = recuperacion.buscar_en_papers.invoke({"consulta": pregunta, "titulo_paper": "subliminal"})
    con_contexto = modulo_agente.invocar_llm(llm, [
        SystemMessage("Respondé en dos oraciones usando solo este contexto y citá la página:\n\n" + contexto),
        HumanMessage(pregunta),
    ])
    display(Markdown(f"**Sin contexto:** {sin_contexto.content}"))
    display(Markdown(f"**Con contexto:** {con_contexto.content}"))
else:
    print("Celda omitida: EJECUTAR_LLM = False")

**Sin contexto:** El modelo maestro debe ser capaz de generar datos o etiquetas de alta calidad que el modelo estudiante no puede obtener directamente de los datos crudos. Además, el modelo estudiante debe ser entrenado para imitar las predicciones o el comportamiento del maestro, transfiriendo así el conocimiento subliminalmente sin necesidad de etiquetas explícitas.

**Con contexto:** Para que ocurra el subliminal learning, el modelo maestro y el modelo estudiante deben compartir inicializaciones similares, lo que implica que generalmente se derivan del mismo modelo de referencia (pág. 1, pág. 2).

## 8. Pruebas del chatbot

Cada caso declara qué tools se esperan. La columna `cumple` compara ese valor con lo que decidió el agente.
La última pregunta sigue a la anterior en la misma conversación.

In [15]:
def preguntar(texto, thread_id, user_id="demo"):
    """Envía un mensaje al agente y devuelve la respuesta, las tools del turno y el tiempo."""
    inicio = time.time()
    sesion = {"configurable": {"thread_id": thread_id, "user_id": user_id}}
    mensajes = agente.invoke({"messages": [("user", texto)]}, sesion)["messages"]
    return mensajes[-1].content, chatbot._herramientas_usadas(mensajes), round(time.time() - inicio)


casos = [
    ("small-talk", "Hola! cómo andás?", set(), "c1"),
    ("general", "¿Cuál es la capital de Francia?", set(), "c2"),
    ("papers ES", "¿Cuántos parámetros tiene GPT-3?", {"buscar_en_papers"}, "c3"),
    ("papers EN", "What does the Scaling Laws paper say about model size and loss?", {"buscar_en_papers"}, "c4"),
    ("resumen", 'Resumir el artículo "Attention Is All You Need" en tres párrafos.', {"buscar_en_papers"}, "c5"),
    ("sin evidencia", "¿Qué dice el paper de Attention sobre el mundial de fútbol 2022?", {"buscar_en_papers"}, "c6"),
    ("estadística", "¿Cuántos correos de phishing hay en el dataset?", {"estadisticas_dataset"}, "c7"),
    ("ranking", "¿Qué dominios usan más los correos de phishing?", {"ranking_dataset"}, "c8"),
    ("dataset", "Mostrame un ejemplo de correo de phishing bancario", {"buscar_en_dataset"}, "c9"),
    ("análisis", "Este correo dice 'Your account is suspended, click here'. ¿Es phishing?", {"buscar_en_dataset"}, "c10"),
    ("contexto", "¿Qué es multi-head attention?", {"buscar_en_papers"}, "c11"),
    ("seguimiento", "¿Y cuántas cabezas usa el modelo big?", {"buscar_en_papers"}, "c11"),
]

resultados = []
if EJECUTAR_LLM:
    for tipo, texto, esperadas, thread_id in casos:
        try:
            respuesta, usadas, segundos = preguntar(texto, thread_id)
            error = ""
        except Exception as excepcion:
            respuesta, usadas, segundos, error = "", [], 0, type(excepcion).__name__
        resultados.append({"tipo": tipo, "pregunta": texto, "tools": sorted(set(usadas)), "cumple": set(usadas) == esperadas,
                           "segundos": segundos, "error": error, "respuesta": respuesta})
        display(Markdown(f"**[{tipo}] {texto}**\n\n*Tools: {sorted(set(usadas)) or 'ninguna'}*\n\n{respuesta or error}\n\n---"))
else:
    print("Celda omitida: EJECUTAR_LLM = False")

**[small-talk] Hola! cómo andás?**

*Tools: ninguna*

¡Hola! Muy bien, gracias. ¿Y vos? ¿En qué te puedo ayudar hoy?

---

**[general] ¿Cuál es la capital de Francia?**

*Tools: ninguna*

La capital de Francia es París.

(Nota: esto es conocimiento general, no sale de los documentos cargados.)

---

**[papers ES] ¿Cuántos parámetros tiene GPT-3?**

*Tools: ['buscar_en_papers']*

GPT-3 tiene **175 mil millones (175B) de parámetros** (Language Models are Few-Shot Learners, pág. 40).

Como dato complementario, el paper también entrena una serie de modelos más pequeños, de 125 millones a 13 mil millones de parámetros, para comparar su rendimiento en los escenarios zero-shot, one-shot y few-shot (pág. 6).

---

**[papers EN] What does the Scaling Laws paper say about model size and loss?**

*Tools: ['buscar_en_papers']*

According to *Scaling Laws for Neural Language Models* (Kaplan et al.):

- **Power-law relationship**: The cross-entropy loss scales as a power-law with model size (N), dataset size (D), and compute used for training, with trends spanning more than seven orders of magnitude (pág. 1).

- **Predictable loss vs. model size**: The early-stopped test loss L(N, D) varies predictably with both dataset size and model size according to Equation (1.5). After an initial transient period, learning curves for all model sizes can be fit with Equation (1.6), parameterized by S_min (the number of steps when training at large batch size). These relations hold across six orders of magnitude in N (pág. 5).

- **Larger models are more sample-efficient**: Larger models train faster and reach fixed loss values with fewer examples/steps (pág. 24).

- **Dataset size can grow sub-linearly in model size** while avoiding overfitting, though this doesn't always represent maximally compute-efficient training (pág. 12).

- **Suboptimal model sizes**: Models between 0.6x and 2.2x the optimal size can be used with only a ~20% increase in compute budget to reach a given loss (pág. 22).

- **No solid theory yet**: The authors note they don't have a solid theoretical understanding for the scaling laws, especially the relations with model size and compute (pág. 22).

- **Architecture details matter little**: Network width or depth have minimal effects within a wide range (pág. 1).

---

**[resumen] Resumir el artículo "Attention Is All You Need" en tres párrafos.**

*Tools: ['buscar_en_papers']*

**Resumen de "Attention Is All You Need" (Vaswani et al.)**

El paper propone el **Transformer**, una nueva arquitectura de red neuronal para transducción de secuencias basada **exclusivamente en mecanismos de atención**, eliminando por completo la recurrencia y las convoluciones que dominaban los modelos encoder-decoder previos. Esta decisión permite una paralelización mucho mayor y un entrenamiento significativamente más rápido: el modelo alcanza un nuevo estado del arte en calidad de traducción tras entrenarse apenas 12 horas en ocho GPUs P100 (págs. 1-2).

En cuanto a la arquitectura, el Transformer sigue una estructura encoder-decoder: el **encoder** está compuesto por un apilamiento de N = 6 capas idénticas, cada una con dos subcapas —un mecanismo de **auto-atención multi-cabeza** y una red feed-forward completamente conectada posición a posición— con conexiones residuales y normalización de capa alrededor de cada subcapa (pág. 3). El modelo usa la atención multi-cabeza de tres maneras: en las capas de "atención encoder-decoder" (donde las queries provienen del decoder y las keys/values del encoder, permitiendo que cada posición del decoder atienda a todas las posiciones de la entrada), en las capas de auto-atención del encoder, y en las del decoder (pág. 5).

En los experimentos, el Transformer supera a los mejores modelos previos en dos tareas de traducción automática: logra **28.4 BLEU** en WMT 2014 inglés-alemán (mejorando en más de 2 BLEU sobre los mejores resultados existentes, incluidos los ensembles) y establece un nuevo récord single-model de **41.8 BLEU** en WMT 2014 inglés-francés, entrenado 3.5 días en ocho GPUs (pág. 1). Los autores lo presentan como el primer modelo de transducción de secuencias basado enteramente en atención, sin RNNs alineados por secuencia ni convoluciones, y plantean como futuros trabajos extenderlo a otras modalidades (imágenes, audio, video) y estudiar mecanismos de atención local y restringida (págs. 2, 10).

---

**[sin evidencia] ¿Qué dice el paper de Attention sobre el mundial de fútbol 2022?**

*Tools: ['buscar_en_papers']*

No tengo información sobre el mundial de fútbol 2022 en el paper de Attention: los fragmentos recuperados no mencionan nada sobre ese tema.

El paper de "Attention Is All You Need" trata sobre el modelo Transformer y mecanismos de atención para tareas de procesamiento de lenguaje, no sobre fútbol. Si querés, puedo buscar en el paper algún tema relacionado con su contenido (arquitectura, entrenamiento, resultados, etc.).

---

... (2 outputs left out)

**[dataset] Mostrame un ejemplo de correo de phishing bancario**

*Tools: ['buscar_en_dataset']*

No encontré un ejemplo de phishing **bancario** específico en el dataset. Los fragmentos recuperados son de phishing, pero de otros tipos:

1. **Phishing de crédito** (corpus SpamAssasin): remitente "HILARIA CHUNG" <alksjdwieytgfbn@msn.com>, asunto *"I never sent you this - keep it hush hush! :)"*. Promete "reparar tu crédito online" y pide hacer clic en un enlace. Señales típicas: remitente personal (msn.com), promesas exageradas y texto con caracteres extraños.

2. **Phishing de contraseña** (corpus Nazario): remitente Lawrence McDevitt <lmcdevitt@ohlone.edu>, asunto *"RE: HELP_DESK_SERVICE"*. Dice "Your Password Has Expired" y pide "CLICK HERE To Verify". Señales: urgencia, enlace genérico y remitente que no coincide con un servicio real.

3. **Phishing de verificación de cuenta** (corpus Nazario): remitente Email Admin <noreply@accountadministrator.com>, asunto *"(Warning)Please Verify Your Account"*. Avisa de "actividad inusual" y pide completar una verificación por un enlace. Señales: dominio sospechoso (accountadministrator.com), trato genérico y presión.

4. **Spam/venta de herramientas maliciosas** (corpus Nazario): remitente support <dgps.support@mahaonline.gov.in>. Ofrece "BANK LOGING PAGE", cuentas, CC/CVV, etc. No es un phishing bancario dirigido, sino la venta de herramientas para atacar bancos.

Si querés, puedo buscar de nuevo con otras palabras (por ejemplo "bank account", "wire transfer", "verify your bank") para intentar encontrar un caso más cercano al phishing bancario.

---

**[análisis] Este correo dice 'Your account is suspended, click here'. ¿Es phishing?**

*Tools: ['buscar_en_dataset']*

Sí, es muy probable que sea phishing. El texto "Your account is suspended, click here" coincide con patrones típicos de phishing que aparecen en el dataset:

- **Urgencia y amenaza**: "account will be suspended/locked", "without further notice", "you will lose your email" (correos phishing, corpus Nazario).
- **Llamado a la acción con enlace**: "Click here to verify now", "Kindly Click here to verify" (corpus Nazario).
- **Remitentes falsos o genéricos**: "Email service", "Email-Host Security", "Notification Service", o incluso suplantando marcas conocidas como Google AdWords (corpus Nazario y CEAS_08).

Señales de alerta en tu correo:
1. Amenaza de suspensión para generar pánico.
2. Enlace "click here" sin contexto claro.
3. Probablemente un remitente genérico o que suplanta a un servicio.

Recomendación: no hagas clic en el enlace. Verificá el estado de tu cuenta ingresando directamente al sitio oficial del servicio, y reportá el correo como phishing.

¿Querés que busque más ejemplos de este tipo de patrón o que compare con correos legítimos similares?

---

**[contexto] ¿Qué es multi-head attention?**

*Tools: ['buscar_en_papers']*

**Multi-head attention** es una variante del mecanismo de atención que, en lugar de usar una sola función de atención con claves, valores y consultas de dimensión completa (d_model), proyecta linealmente las queries, keys y values **h veces** con proyecciones lineales aprendidas distintas, hacia dimensiones d_k, d_k y d_v respectivamente. Sobre cada versión proyectada se aplica la atención de forma independiente, y luego los resultados se concatenan y se proyectan una vez más para obtener la salida final:

MultiHead(Q, K, V) = Concat(head₁, ..., head_h) W^O, donde head_i = Attention(QW_i^Q, KW_i^K, VW_i^V).

Puntos clave según el paper *Attention Is All You Need*:

- **Propósito**: permite al modelo atender conjuntamente a información de **diferentes subespacios de representación** y en **diferentes posiciones**. Con un solo head, el promedio (averaging) inhibe esta capacidad (pág. 5).
- **Configuración usada**: h = 8 cabezas en paralelo, con d_k = d_v = d_model/h = 64 (pág. 5).
- **Costo computacional**: gracias a la dimensión reducida de cada head, el costo total es similar al de una atención single-head con dimensión completa (pág. 5).
- **Evidencia de que cada head aprende tareas distintas**: las visualizaciones muestran que los heads aprenden a realizar tareas diferentes; por ejemplo, varios heads atienden a dependencias de larga distancia del verbo "making" para completar la frase "making...more difficult" (págs. 13–15).

En resumen: es como correr varias "atenciones" en paralelo, cada una enfocada en un subespacio distinto, y combinar sus salidas.

---

**[seguimiento] ¿Y cuántas cabezas usa el modelo big?**

*Tools: ['buscar_en_papers']*

El modelo **big** usa **h = 16 cabezas** (con d_model = 1024, d_ff = 4096, N = 6 capas, P_drop = 0.3 y 300K pasos de entrenamiento), según la Tabla 3 del paper *Attention Is All You Need* (pág. 9).

Como referencia, el modelo base usa h = 8 cabezas con d_model = 512 (pág. 5).

---

In [16]:
tabla_resultados = pd.DataFrame(resultados)
if EJECUTAR_LLM:
    print(f"Decisión de tools correcta: {tabla_resultados['cumple'].sum()} de {len(tabla_resultados)}")
    print(f"Tiempo medio por pregunta: {tabla_resultados['segundos'].mean():.0f} s")
    display(tabla_resultados[["tipo", "tools", "cumple", "segundos", "error"]])

Decisión de tools correcta: 12 de 12
Tiempo medio por pregunta: 33 s


,tipo,tools,cumple,segundos,error
0,small-talk,[],True,1,
1,general,[],True,0,
2,papers ES,[buscar_en_papers],True,17,
3,papers EN,[buscar_en_papers],True,47,
4,resumen,[buscar_en_papers],True,51,
5,sin evidencia,[buscar_en_papers],True,33,
6,estadística,[estadisticas_dataset],True,36,
7,ranking,[ranking_dataset],True,35,
8,dataset,[buscar_en_dataset],True,41,
9,análisis,[buscar_en_dataset],True,38,


## 9. Memoria y sesiones

- **Corto plazo:** el checkpointer guarda cada conversación por `thread_id`. Cada llamada al LLM recibe una
  ventana de los últimos mensajes y el turno actual completo.
- **Largo plazo:** el agente decide qué guardar con `guardar_recuerdo`. Los recuerdos se guardan por `user_id`
  y se agregan al prompt en cada turno.
- **Sesiones:** cada invocación lleva su `thread_id` y su `user_id`.

In [17]:
def decir(usuario, conversacion, texto):
    respuesta, usadas, _ = preguntar(texto, f"{usuario}-{conversacion}", usuario)
    display(Markdown(f"**{usuario} / conversación {conversacion}:** {texto}\n\n{respuesta}\n\n*Tools: {usadas or 'ninguna'}*"))


if EJECUTAR_LLM:
    decir("ana", "1", "Hola! Soy Ana y trabajo en seguridad informática en un banco.")
    decir("ana", "1", "En una frase: ¿qué es el few-shot learning?")
    print("Recuerdos guardados de Ana:", memoria.leer_recuerdos(agente.store, "ana"))
    decir("ana", "2", "¿Te acordás cómo me llamo y a qué me dedico?")
    decir("ana", "2", "¿Qué te pregunté en la otra conversación?")
else:
    print("Celda omitida: EJECUTAR_LLM = False")

**ana / conversación 1:** Hola! Soy Ana y trabajo en seguridad informática en un banco.

¡Hola, Ana! Un gusto saludarte. Qué bueno que trabajes en seguridad informática en un banco.

¿En qué te puedo ayudar hoy?

*Tools: ['guardar_recuerdo']*

**ana / conversación 1:** En una frase: ¿qué es el few-shot learning?

El few-shot learning es una capacidad de los modelos de lenguaje (como GPT) de aprender a realizar una tarea nueva a partir de solo unos pocos ejemplos de entrada-salida proporcionados en el propio prompt, sin necesidad de un entrenamiento adicional.

(Nota: esto es conocimiento general, no sale de los documentos cargados.)

*Tools: ninguna*

Recuerdos guardados de Ana: ['Se llama Ana y trabaja en seguridad informática en un banco.']


**ana / conversación 2:** ¿Te acordás cómo me llamo y a qué me dedico?

¡Claro que sí, Ana! Recordás que trabajás en seguridad informática en un banco. ¿En qué te puedo ayudar hoy?

*Tools: ninguna*

**ana / conversación 2:** ¿Qué te pregunté en la otra conversación?

No tengo acceso a conversaciones anteriores: solo conservo los datos duraderos que guardé en la memoria a largo plazo, que en tu caso es que te llamás Ana y que trabajás en seguridad informática en un banco.

No guardo el historial de preguntas puntuales de otras charlas, así que no puedo recordarte qué me preguntaste antes. Si querés retomar algún tema (papers, dataset de correos, análisis de un correo sospechoso, etc.), decime y lo retomamos desde cero.

*Tools: ninguna*

### Pausar y retomar después de reiniciar

Se crea un agente nuevo, con conexiones nuevas a SQLite y sin nada en memoria RAM, y Ana vuelve a la
conversación 1.

In [18]:
if EJECUTAR_LLM:
    agente = modulo_agente.construir_agente(memoria.crear_memoria_corto_plazo(), memoria.crear_memoria_largo_plazo())
    decir("ana", "1", "¿Qué te había preguntado recién?")
else:
    print("Celda omitida: EJECUTAR_LLM = False")

**ana / conversación 1:** ¿Qué te había preguntado recién?

Recién me preguntaste, en una frase, qué es el few-shot learning.

*Tools: ninguna*

### Sesiones de usuario en paralelo

In [19]:
if EJECUTAR_LLM:
    with ThreadPoolExecutor() as hilos:
        tareas = {
            usuario: hilos.submit(preguntar, "Hola, ¿sabés cómo me llamo?", f"{usuario}-paralelo", usuario)
            for usuario in ("ana", "beto")
        }
    for usuario, tarea in tareas.items():
        print(f"{usuario}: {tarea.result()[0]}")
    print("Recuerdos de Beto:", memoria.leer_recuerdos(agente.store, "beto"))
else:
    print("Celda omitida: EJECUTAR_LLM = False")

ana: ¡Hola, Ana! Sí, sé que te llamás Ana. ¿En qué te puedo ayudar hoy?
beto: ¡Hola! No, todavía no sé cómo te llamás: no tengo recuerdos guardados tuyos. Si querés, decime tu nombre y lo guardo para las próximas conversaciones. 😊
Recuerdos de Beto: []


## 10. Chatbot

`main.py` ejecuta este mismo chatbot por consola. La celda siguiente lo usa con mensajes escritos de antemano,
para que el notebook se pueda ejecutar sin intervención. Para usarlo de forma interactiva, ejecutar
`chatbot.iniciar_chat()`.

In [20]:
def chat_guionado(entradas):
    """Ejecuta el chatbot real reemplazando input() por una lista de mensajes."""
    pendientes = iter(entradas)

    def responder(indicador=""):
        texto = next(pendientes)
        print(f"{indicador}{texto}")
        return texto

    original, builtins.input = builtins.input, responder
    try:
        chatbot.iniciar_chat()
    finally:
        builtins.input = original


if EJECUTAR_LLM:
    usuario = f"demo-{uuid.uuid4().hex[:6]}"
    chat_guionado([f"/usuario {usuario}", "Hola, me llamo Sofía", "/nueva", "Gracias, ¿qué podés hacer?", "/conversaciones", "/salir"])
    conversaciones = memoria.listar_conversaciones(memoria.crear_memoria_largo_plazo(), usuario)
    primera = next(iter(conversaciones))
    chat_guionado([f"/usuario {usuario}", f"/retomar {primera}", "¿Cómo me llamo?", "/salir"])
else:
    print("Celda omitida: EJECUTAR_LLM = False")

Cargando modelo y memorias (puede tardar unos segundos la primera vez)...



=== Agentic RAG — Chatbot (Taller de Tecnologías 2) ===
Comandos disponibles:
  /nueva          -> pausa esta conversación y empieza una nueva
  /conversaciones -> lista tus conversaciones guardadas
  /retomar <id>   -> vuelve a una conversación guardada, con todo su contexto
  /usuario <id>   -> cambia de usuario (otros recuerdos de largo plazo) y empieza conversación nueva
  /ayuda          -> muestra esta ayuda
  /salir          -> termina el programa

Podés preguntar en español o inglés sobre:
  - los papers cargados (Attention, GPT, GPT-3, Scaling Laws, Subliminal Learning,
    Designing Data-Intensive Applications),
  - el dataset de correos de phishing/legítimos (ejemplos, patrones, estadísticas).
Si contás algo sobre vos (nombre, trabajo, intereses), el asistente puede recordarlo.

[usuario · conv b78d598d] > /usuario demo-9b779c
(ahora sos 'demo-9b779c', conversación nueva: 9723a079)

[demo-9b779c · conv 9723a079] > Hola, me llamo Sofía



¡Hola, Sofía! 👋 Qué gusto saludarte.

Soy tu asistente en ciberseguridad (phishing) y en los papers de la materia (Attention, GPT, GPT-3, Scaling Laws, Subliminal Learning y Designing Data-Intensive Applications).

¿En qué te puedo ayudar hoy? Podés preguntarme por:
- **Los papers**: conceptos, métodos, resultados.
- **El dataset de correos**: ejemplos de phishing o legítimos, estadísticas, remitentes o dominios más frecuentes.
- **Un correo sospechoso**: puedo compararlo con casos conocidos.

Decime por dónde empezamos. 😊
  (usó: guardar_recuerdo)

[demo-9b779c · conv 9723a079] > /nueva
(nueva conversación: e119b919)

[demo-9b779c · conv e119b919] > Gracias, ¿qué podés hacer?



¡De nada, Sofía! Puedo ayudarte con:

- **Papers de la materia**: responder preguntas sobre los documentos cargados (Transformers/Attention, GPT, GPT-3, Scaling Laws, Subliminal Learning y *Designing Data-Intensive Applications*), con citas de página.
- **Dataset de correos**: buscar ejemplos reales de phishing o legítimos, comparar un correo sospechoso con casos conocidos, y darte estadísticas (totales, composición) o rankings (remitentes, dominios o asuntos más frecuentes).
- **Ciberseguridad en general**: dudas sobre phishing y seguridad, con mi conocimiento general (te aclaro cuando algo no sale de los documentos).
- **Memoria**: puedo recordar datos tuyos (nombre, trabajo, intereses) para futuras conversaciones.

¿Por dónde querés empezar?

[demo-9b779c · conv e119b919] > /conversaciones
  9723a079: Hola, me llamo Sofía
  e119b919 (actual): Gracias, ¿qué podés hacer?

[demo-9b779c · conv e119b919] > /salir
¡Hasta luego!
Cargando modelo y memorias (puede tardar unos segundos la pr


=== Agentic RAG — Chatbot (Taller de Tecnologías 2) ===
Comandos disponibles:
  /nueva          -> pausa esta conversación y empieza una nueva
  /conversaciones -> lista tus conversaciones guardadas
  /retomar <id>   -> vuelve a una conversación guardada, con todo su contexto
  /usuario <id>   -> cambia de usuario (otros recuerdos de largo plazo) y empieza conversación nueva
  /ayuda          -> muestra esta ayuda
  /salir          -> termina el programa

Podés preguntar en español o inglés sobre:
  - los papers cargados (Attention, GPT, GPT-3, Scaling Laws, Subliminal Learning,
    Designing Data-Intensive Applications),
  - el dataset de correos de phishing/legítimos (ejemplos, patrones, estadísticas).
Si contás algo sobre vos (nombre, trabajo, intereses), el asistente puede recordarlo.

[usuario · conv 951d08fc] > /usuario demo-9b779c
(ahora sos 'demo-9b779c', conversación nueva: 3c63ff63)

[demo-9b779c · conv 3c63ff63] > /retomar 9723a079
(conversación retomada: Hola, me llamo Sof


Te llamás **Sofía**. 😊

¿En qué te puedo ayudar hoy?

[demo-9b779c · conv 9723a079] > /salir
¡Hasta luego!


## 11. Limitaciones

- **Cuota de Groq:** el plan gratuito permite 8.000 tokens por minuto y 200.000 por día. Una pregunta con
  búsqueda hace al menos dos llamadas y puede tardar decenas de segundos.
- **Largo de las respuestas:** están limitadas a 900 tokens de salida.
- **Resúmenes:** se generan con hasta 10 fragmentos recuperados, no con el documento completo.
- **Páginas:** las citas usan el número de página del PDF, que puede diferir del número impreso.
- **Control de alucinaciones:** se apoya en el prompt, el umbral y las citas, por lo que reduce el riesgo pero
  no lo elimina.

## 12. Experimentos opcionales

La letra propone cuatro opcionales que suman si se hacen bien. Esta sección desarrolla los dos que
corresponden a temas vistos en clase (presentaciones de Embeddings, Bases de datos vectoriales y RAG):

- **Estrategias de chunking y de recuperación** (12.2 y 12.3).
- **Comparación de modelos de embeddings** (12.4).

Los experimentos no usan el LLM: no gastan cuota de Groq y dan siempre el mismo resultado. El código
está en `src/experimentos.py`.

### 12.1 Metodología

**Conjunto de evaluación.** 20 preguntas sobre los cinco papers (Attention, GPT, GPT-3, Scaling Laws y
Subliminal Learning), cuatro por paper. Para cada una se buscó en el PDF la página donde está la
respuesta. Cada pregunta está en español y en inglés, porque los papers están en inglés y el chatbot
tiene que entender preguntas en español.

**Métricas.** Para cada configuración se recuperan los 4 fragmentos más parecidos (el mismo `k` que usa
el agente):

- **Acierto@4:** porcentaje de preguntas en las que al menos un fragmento recuperado viene de una página
  correcta. Si no hay ninguno, el LLM no tiene con qué responder.
- **MRR** (*Mean Reciprocal Rank*): promedio de 1/posición del primer fragmento correcto. Vale 1 si el
  correcto sale siempre primero, 0,5 si en promedio sale segundo y 0 si nunca aparece.

**Condiciones.** Para que las configuraciones sean comparables:

- Se usa el mismo preprocesamiento que la ingesta del proyecto (`normalizar_texto` y el filtro de
  fragmentos con menos de 50% de letras) y se fragmenta página por página, así cada fragmento conserva
  su número de página.
- Las colecciones de Chroma se crean en memoria; `chroma_db/` no se modifica.
- Se deja afuera el libro de Kleppmann (600 páginas), que haría cada configuración unas cuatro veces
  más lenta.
- Todos los modelos devuelven vectores normalizados y Chroma usa distancia L2, como el proyecto. Con
  vectores normalizados, L2 y coseno ordenan igual (L2² = 2 − 2·coseno), así que los puntajes quedan en
  la misma escala que el umbral del proyecto.

**Cómo leer los resultados.** Con 20 preguntas, cada pregunta vale 5 puntos porcentuales. Una
diferencia de 5 o 10 puntos es una o dos preguntas y no alcanza para afirmar que una configuración es
mejor. Además, el criterio por página es estricto: si un fragmento de otra página también menciona el
dato, cuenta como fallo.

In [21]:
import os
import sys
import time
import warnings
from pathlib import Path

# Salidas más limpias: sin barras de progreso al cargar modelos ni avisos de Hugging Face.
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HUB_VERBOSITY"] = "error"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
# Los modelos MiniLM dan puntajes negativos y LangChain lo advierte (ver conclusiones de 12.4).
warnings.filterwarnings("ignore", message="Relevance scores must be between 0 and 1")

import pandas as pd
import transformers

transformers.logging.disable_progress_bar()
sys.path.insert(0, str(Path.cwd()))
from src import experimentos

pd.set_option("display.precision", 2)
paginas = experimentos.cargar_paginas()
modelo_proyecto = experimentos.crear_modelo("multilingual-e5-small")
print(f"Páginas cargadas: {len(paginas)} de {len(experimentos.PAPERS)} papers")
pd.DataFrame([
    {"paper": p.archivo.split(" (")[0][:45], "páginas con la respuesta": p.paginas, "pregunta (ES)": p.es}
    for p in experimentos.PREGUNTAS
])

Páginas cargadas: 166 de 5 papers


,paper,páginas con la respuesta,pregunta (ES)
0,Attention is all you need.pdf,"(5,)",¿Cuántas cabezas de atención en paralelo usa e...
1,Attention is all you need.pdf,"(7,)",¿Qué optimizador se usó para entrenar el Trans...
2,Attention is all you need.pdf,"(8,)",¿Qué valor de label smoothing se usó al entren...
3,Attention is all you need.pdf,"(7,)",¿Cuántos pares de oraciones tiene el dataset W...
4,Attention is all you need.pdf,"(6,)",¿Cómo codifica el Transformer la posición de l...
5,Attention is all you need.pdf,"(1, 8)",¿Qué puntaje BLEU obtuvo el Transformer en ing...
6,GPT Improving Language Understanding by Gener,"(4, 5)",¿Con qué corpus de libros se pre-entrenó GPT?
7,GPT Improving Language Understanding by Gener,"(5,)",¿Cuántas capas y cabezas de atención tiene el ...
8,GPT Improving Language Understanding by Gener,"(8,)",¿Ayuda el objetivo auxiliar de modelado de len...
9,GPT Improving Language Understanding by Gener,"(4,)",¿Cómo adapta GPT las entradas estructuradas a ...


### 12.2 Estrategias de chunking

Se comparan las estrategias vistas en la presentación de RAG, con el mismo modelo (e5-small) y la
misma recuperación (top-4):

| Estrategia | Qué hace |
|---|---|
| Recursivo 250/30, 500/50, 800/100 y 1200/200 | Corta por tamaño, intentando respetar párrafos y oraciones. El segundo número es el solapamiento (*overlap*). 800/100 es el del proyecto. |
| Recursivo 800/0 | Igual que el del proyecto, sin solapamiento, para medir cuánto aporta. |
| Por oraciones | Agrupa oraciones completas hasta 800 caracteres. Nunca corta una oración al medio. |
| Semántico | Calcula el embedding de cada oración y empieza un fragmento nuevo donde la similitud con la siguiente oración cae al 20% más bajo, es decir, donde cambia el tema. |

In [22]:
inicio = time.time()
tabla_chunking = experimentos.experimento_chunking(paginas, modelo_proyecto)
print(f"Tiempo total: {time.time() - inicio:.0f} s")
tabla_chunking

Tiempo total: 219 s


,estrategia,fragmentos,largo medio,acierto ES,MRR ES,acierto EN,MRR EN,indexado (s)
0,recursivo 250/30,2231,237,0.45,0.40,0.70,0.52,23.4
1,recursivo 500/50,1130,464,0.55,0.46,0.65,0.53,27.6
2,recursivo 800/100 (proyecto),748,720,0.45,0.36,0.70,0.45,28.0
3,recursivo 800/0 (sin solapamiento),679,705,0.55,0.44,0.60,0.52,25.7
4,recursivo 1200/200,538,1038,0.60,0.46,0.60,0.54,30.0
5,por oraciones (hasta 800),741,660,0.45,0.42,0.65,0.49,25.7
6,"semántico (e5, percentil 20)",951,479,0.60,0.46,0.50,0.42,25.6


**Conclusiones sobre el chunking**

- **Ninguna estrategia gana con claridad.** Las diferencias entre estrategias son de una a tres
  preguntas y cambian de signo según el idioma: el semántico y el de 1200/200 son los mejores en español
  (60%), pero el semántico es el peor en inglés (50%). Con este conjunto no hay evidencia para cambiar.
- **El tamaño cambia el costo más que el acierto.** Con fragmentos de 1200 caracteres, cada búsqueda
  le manda al LLM unos 1.300 caracteres más que con 800. Con el límite de 8.000 tokens por minuto de
  Groq eso pesa, y no se traduce en una mejora clara de recuperación.
- **El solapamiento no mostró beneficio medible** (800/0 contra 800/100). Se mantiene porque evita
  cortar una idea justo en el borde y su costo es bajo: unos 70 fragmentos más.
- **El chunking semántico es más caro de construir:** hay que calcular el embedding de cada oración
  antes de cortar, y deja fragmentos de largo muy variable.
- **Decisión:** se mantiene el corte recursivo de 800/100. Cambiarlo implicaría reindexar los 33.792
  fragmentos del dataset sin una mejora demostrada.

### 12.3 Estrategias de recuperación

Sobre el índice del proyecto (800/100 con e5-small) se compara cómo se eligen los fragmentos:

| Recuperación | Qué hace |
|---|---|
| top-4 | Los 4 fragmentos más parecidos (kNN). |
| top-8 | Los 8 más parecidos. Más contexto, pero el doble de tokens por búsqueda. |
| top-4 + umbral | Lo que hace la tool del proyecto: descarta los fragmentos con relevancia menor a 0,68. |
| MMR | *Maximal Marginal Relevance*: de los 20 más parecidos elige 4 que sean relevantes pero distintos entre sí, para no repetir información. |
| top-4 + filtro por paper | Busca solo dentro del paper correcto. Simula cuando el agente completa `titulo_paper` en `buscar_en_papers`. |

`LongContextReorder` no se compara acá porque no cambia qué fragmentos se recuperan, solo el orden en
que se le pasan al LLM. Para medir su efecto habría que evaluar las respuestas del LLM.

In [23]:
tabla_recuperacion = experimentos.experimento_recuperacion(paginas, modelo_proyecto)
tabla_recuperacion

,recuperación,acierto ES,MRR ES,acierto EN,MRR EN
0,top-4 (similitud),0.45,0.36,0.70,0.45
1,top-8 (similitud),0.60,0.39,0.80,0.47
2,top-4 + umbral 0.68 (proyecto),0.45,0.36,0.70,0.45
3,"MMR (k=4, fetch_k=20, λ=0,5)",0.55,0.39,0.55,0.41
4,top-4 + filtro por paper,0.60,0.45,0.85,0.54


Para entender las fallas, la celda siguiente muestra, para cada pregunta que el top-4 no encuentra,
de qué paper y página vienen los fragmentos recuperados.

In [24]:
indice, _ = experimentos.indexar(experimentos.chunking_recursivo(paginas, 800, 100), modelo_proyecto)
fallas = []
for pregunta in experimentos.PREGUNTAS:
    for idioma in ("es", "en"):
        documentos = indice.similarity_search(getattr(pregunta, idioma), k=experimentos.K)
        if not any(experimentos._es_correcto(d, pregunta) for d in documentos):
            fallas.append({
                "idioma": idioma.upper(),
                "pregunta": getattr(pregunta, idioma)[:60],
                "esperado": f"{pregunta.archivo[:14]} {pregunta.paginas}",
                "recuperado (paper, página)": ", ".join(
                    f"{d.metadata['archivo'][:10]} p{d.metadata['pagina']}" for d in documentos),
            })
indice.delete_collection()
print(f"Fallas: {len(fallas)} de {2 * len(experimentos.PREGUNTAS)} búsquedas")
pd.set_option("display.max_colwidth", 90)
pd.DataFrame(fallas)

Fallas: 17 de 40 búsquedas


,idioma,pregunta,esperado,"recuperado (paper, página)"
0,ES,¿Cuántas cabezas de atención en paralelo usa el Transformer?,"Attention is a (5,)","Attention p1, Attention p1, Attention p2, Attention p8"
1,ES,¿Qué optimizador se usó para entrenar el Transformer?,"Attention is a (7,)","Scaling La p1, Attention p1, Attention p8, Scaling La p1"
2,EN,Which optimizer was used to train the Transformer?,"Attention is a (7,)","Scaling La p1, Attention p1, Scaling La p1, Attention p8"
3,ES,¿Cómo codifica el Transformer la posición de las palabras?,"Attention is a (6,)","Attention p1, Attention p9, Attention p1, Attention p8"
4,EN,How does the Transformer encode the position of the tokens?,"Attention is a (6,)","Attention p3, GPT Improv p3, Attention p5, Scaling La p9"
5,ES,¿Con qué corpus de libros se pre-entrenó GPT?,"GPT Improving (4, 5)","GPT Improv p9, Language M p16, Language M p34, Attention p8"
6,EN,Which book corpus was GPT pre-trained on?,"GPT Improving (4, 5)","2507.14805 p5, GPT Improv p9, Language M p6, Language M p34"
7,ES,¿Cuántas capas y cabezas de atención tiene el modelo GPT?,"GPT Improving (5,)","Language M p6, Language M p8, Language M p1, Language M p40"
8,EN,How many layers and attention heads does the GPT model have?,"GPT Improving (5,)","Language M p8, Language M p8, Attention p10, Attention p5"
9,ES,¿Ayuda el objetivo auxiliar de modelado de lenguaje durante,"GPT Improving (8,)","Language M p1, Language M p5, 2507.14805 p7, GPT Improv p3"


**Conclusiones sobre la recuperación**

- **El filtro por paper es lo que más mejora** (de 45% a 60% en español y de 70% a 85% en inglés, con el
  mejor MRR). La tabla de fallas muestra por qué: varias búsquedas sobre GPT terminan en GPT-3 y al
  revés, y otras caen en páginas de otro paper que tratan el mismo tema. Esto respalda dos decisiones del
  proyecto: el parámetro `titulo_paper` de `buscar_en_papers` y la resolución de nombres aproximados
  (`PATRONES_PAPERS`), que permite que el filtro funcione aunque el LLM escriba "gpt3" o "transformer".
- **La página 1 atrae muchas búsquedas.** El resumen y la introducción de cada paper mencionan todos sus
  temas, así que se parecen a muchas preguntas pero no tienen el dato puntual.
- **El umbral de 0,68 no perdió ninguna pregunta pertinente** (mismos resultados que top-4). Es decir,
  solo corta lo ajeno, que es su objetivo; en 12.4 se mide cuánto de lo ajeno corta.
- **Top-8 mejora el acierto, pero duplica los tokens de cada búsqueda.** Con la cuota de Groq conviene
  más que el agente busque de nuevo con otras palabras o con el filtro, cosa que el prompt ya le indica.
- **MMR no es una mejora estable:** gana dos preguntas en español y pierde tres en inglés. Al buscar
  variedad desplaza fragmentos parecidos entre sí, y a veces el desplazado era el de la página correcta.
- **Las preguntas en español aciertan bastante menos que las mismas preguntas en inglés** (45% contra
  70% en top-4). Como los papers están en inglés, una mejora barata sería indicarle al agente en el
  prompt que escriba la `consulta` de la tool en inglés. No se implementó porque hay que validarla con
  el LLM, y eso gasta cuota.

### 12.4 Modelos de embeddings

Con el mismo chunking (800/100) y la misma recuperación (top-4) se comparan cuatro modelos:

| Modelo | Por qué se incluye |
|---|---|
| `all-MiniLM-L6-v2` | El de los labs del curso. Entrenado en inglés. |
| `paraphrase-multilingual-MiniLM-L12-v2` | Multilingüe, entrenado para detectar paráfrasis. |
| `multilingual-e5-small` | El del proyecto. Multilingüe, con prefijos `query:` y `passage:`. |
| `multilingual-e5-base` | La versión más grande de e5, para ver si el tamaño compensa. |

Además del acierto se mide cuánto se separan los puntajes de las preguntas pertinentes y los de 8
preguntas ajenas a los papers (recetas, fútbol, playas). Eso decide si un umbral puede cortar lo ajeno
sin perder lo pertinente:

- **puntaje pertinentes / ajenas:** promedio del puntaje del mejor fragmento.
- **separables:** porcentaje de pares (pertinente, ajena) en los que la pertinente tiene mayor puntaje.
  100% significa que existe un umbral que separa todo; 50%, que el puntaje no distingue nada.
- **ajenas cortadas / pertinentes perdidas:** qué pasaría con el umbral actual del proyecto (0,68). Solo
  tiene sentido para e5-small, el modelo con el que se calibró; para los demás habría que calibrar otro.

La primera ejecución descarga los modelos que falten desde Hugging Face (unos 1,7 GB en total).

In [25]:
inicio = time.time()
tabla_embeddings = experimentos.experimento_embeddings(paginas)
print(f"Tiempo total: {time.time() - inicio:.0f} s")
tabla_embeddings

Tiempo total: 168 s


,modelo,comentario,dimensión,acierto ES,MRR ES,acierto EN,MRR EN,puntaje pertinentes,puntaje ajenas,separables,ajenas cortadas,pertinentes perdidas,carga (s),indexado (s)
0,all-MiniLM-L6-v2,"el de los labs, entrenado en inglés",384,0.35,0.15,0.55,0.34,0.27,-0.11,0.99,1.00,1.00,0.9,12.7
1,paraphrase-multilingual-MiniLM-L12-v2,"multilingüe, de paráfrasis",384,0.50,0.37,0.55,0.44,0.53,-0.02,1.00,1.00,0.88,1.6,11.6
2,multilingual-e5-small,el del proyecto,384,0.45,0.36,0.70,0.45,0.80,0.66,0.99,0.75,0.00,3.1,28.0
3,multilingual-e5-base,e5 más grande,768,0.65,0.47,0.70,0.62,0.79,0.65,1.00,0.75,0.00,3.4,99.1


**Conclusiones sobre los embeddings**

- **El modelo de los labs no sirve para este caso.** `all-MiniLM-L6-v2` está entrenado en inglés y en
  español acierta 30%, el peor resultado. Confirma que hacía falta un modelo multilingüe.
- **`multilingual-e5-base` es el que mejor recupera:** 65% en español (cuatro preguntas más que
  e5-small) y el mejor MRR en inglés (0,62 contra 0,45). Es la única diferencia del experimento que
  supera el margen de una o dos preguntas.
- **Pero cuesta bastante más:** vectores del doble de dimensión (768), indexación unas tres veces más
  lenta y un modelo de 1,1 GB contra 470 MB. Llevado al índice completo del proyecto (33.792 fragmentos
  del dataset más el libro), la reindexación pasaría de unos 25 minutos a más de una hora en CPU, y
  `chroma_db/` crecería en proporción.
- **Todos los modelos separan bien lo pertinente de lo claramente ajeno** (separables entre 99% y 100%).
  Lo que cambia es la escala: e5 comprime los puntajes (0,80 contra 0,66), mientras que los MiniLM los
  abren (0,53 contra −0,02). Por eso el umbral depende del modelo y habría que recalibrarlo al cambiar.
  Los MiniLM dan incluso puntajes negativos, que LangChain advierte porque espera valores entre 0 y 1.
- **El umbral del proyecto funciona como se diseñó:** con e5-small corta 6 de las 8 consultas ajenas y
  no pierde ninguna pertinente. Las 2 ajenas que pasan las frena el prompt, que exige verificar que los
  fragmentos respondan la pregunta. Estas consultas ajenas son muy distintas de los papers; en la
  sección 4 se ve que con consultas más cercanas los puntajes se solapan.
- **Decisión:** se mantiene e5-small para esta entrega. e5-base es el mejor candidato si se reindexa,
  pero en inglés rinde igual que e5-small (70%), así que gran parte de su ventaja se obtiene más barato
  con la mejora propuesta en 12.3: escribir la consulta de la tool en inglés.

### 12.5 Reflexión

- **Medir cambió las prioridades.** Antes de los experimentos, el chunking parecía la decisión más
  importante. Los resultados muestran que pesa poco frente a otras dos cosas: buscar dentro del paper
  correcto y el idioma de la consulta.
- **Las decisiones del proyecto quedan respaldadas:** corte recursivo de 800/100, umbral de 0,68 y filtro
  por paper con nombres aproximados. La de e5-small queda justificada por costo, no por calidad: e5-base
  recupera mejor.
- **Mejoras concretas que surgen de los datos**, ordenadas por costo:
  1. Indicar en el prompt que la consulta de `buscar_en_papers` se escriba en inglés. No requiere
     reindexar; falta validarla con el LLM.
  2. Indicar al agente que complete `titulo_paper` siempre que la pregunta nombre un paper.
  3. Pasar a `multilingual-e5-base`, reindexar y recalibrar el umbral.
- **Límites del experimento:** 20 preguntas son pocas para diferencias chicas; mide si se recupera la
  página correcta, no si la respuesta final es buena; y deja afuera el libro de Kleppmann y el dataset de
  correos, donde la recuperación podría comportarse distinto. Ampliar el conjunto de preguntas y medir la
  respuesta del LLM serían los próximos pasos.